# AI-Assisted Parallel Image Processing — Colab setup

## Làm lần đầu theo đúng thứ tự

1. Chọn **Runtime → Change runtime type → T4 GPU**, rồi bấm **Connect**.
2. Mở biểu tượng **chìa khóa (Secrets)** ở thanh trái. Tạo `NGROK_AUTHTOKEN`, dán token lấy tại [ngrok dashboard](https://dashboard.ngrok.com/get-started/your-authtoken) và bật quyền truy cập notebook. Nếu dùng chế độ AI, tạo thêm `OPENAI_API_KEY`; manual mode không cần khóa OpenAI.
3. Chạy từng cell từ trên xuống. Cell setup sẽ clone `main`, cài dependency, tải dataset có kiểm tra checksum và build Release bằng OpenMP/CUDA. Có thể mất vài phút ở lần đầu.
4. Chỉ sang cell tiếp theo khi cell hiện tại kết thúc và không có traceback màu đỏ. Kết quả test hợp lệ phải có `100% tests passed`.
5. Ở mục **Mở Streamlit UI trên Colab**, chạy cell và bấm liên kết **Mở Pixel Lab Streamlit UI**. Không mở `localhost:8501` và không dùng cell `serve_kernel_port_as_iframe`.

Không bấm **Lưu trong GitHub** nếu chỉ chạy thử. Khi Colab ngắt hoặc xóa runtime, phải chạy lại các cell từ đầu.

In [ ]:
!nvidia-smi
!nvcc --version

In [ ]:
REPOSITORY = 'https://github.com/Chicken20145/ai-assisted-parallel-image-processing.git'
GIT_REF = 'codex/fix-colab-streamlit-proxy'  # Đổi thành main sau khi PR #7 merge.

%cd /content
!test -d ai-assisted-parallel-image-processing || git clone --branch {GIT_REF} {REPOSITORY}
%cd /content/ai-assisted-parallel-image-processing
!git fetch origin {GIT_REF}
!git switch {GIT_REF}
!git pull --ff-only origin {GIT_REF}

import subprocess

def show_git_sync():
    local_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
    remote_commit = subprocess.check_output(['git', 'rev-parse', f'origin/{GIT_REF}'], text=True).strip()
    if local_commit != remote_commit:
        raise RuntimeError(f'Colab chưa đồng bộ GitHub: local={local_commit[:7]}, GitHub={remote_commit[:7]}')
    print(f'ĐÃ ĐỒNG BỘ GITHUB — branch: {GIT_REF} — commit: {local_commit[:7]}')

show_git_sync()
!bash scripts/setup_colab.sh

## Cập nhật code khi A/B/C làm song song

Sau khi thành viên A push commit mới lên cùng branch, chạy cell dưới để lấy code mới, build tăng dần và test lại. Không cần tải lại dataset hoặc cài lại package trong cùng runtime.

In [ ]:
%cd /content/ai-assisted-parallel-image-processing
!git fetch origin {GIT_REF}
!git switch {GIT_REF}
!git pull --ff-only origin {GIT_REF}
show_git_sync()
!bash scripts/build_colab.sh
!bash scripts/test_colab.sh

## Mở Streamlit UI trên Colab

Streamlit cần WebSocket nên Colab kernel proxy có thể hiện trang trắng hoặc lỗi 404. Cell dưới sẽ:

1. Kiểm tra core `build-colab/image_pipeline_cli`.
2. Chạy Streamlit nền và kiểm tra health endpoint.
3. Đọc `NGROK_AUTHTOKEN` từ Colab Secrets; nếu chưa có sẽ hỏi bằng ô nhập ẩn.
4. Tạo tunnel HTTPS và hiện liên kết **Mở Pixel Lab Streamlit UI**.

Không ghi token trực tiếp vào cell, không chụp/gửi token và không commit token lên GitHub. URL ngrok là URL công khai trong lúc runtime hoạt động; không chia sẻ URL hoặc tải ảnh nhạy cảm. Khi dùng xong, chạy `ngrok.disconnect(ui_url)` hoặc ngắt runtime.

**Xử lý lỗi:** `FileNotFoundError` → chạy lại setup/build; trang trắng hoặc 404 và cell còn `serve_kernel_port_as_iframe` → đang dùng notebook cũ, hãy mở lại bản mới; lỗi xác thực ngrok → kiểm tra tên secret đúng là `NGROK_AUTHTOKEN` và đã bật quyền notebook; lỗi ứng dụng → xem log bằng `print(open('/tmp/pixel_lab_streamlit.log', encoding='utf-8').read())`.

In [ ]:
import getpass
import os
import subprocess
import sys
import time
from pathlib import Path

import requests
from IPython.display import HTML, display

try:
    from pyngrok import ngrok
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyngrok'])
    from pyngrok import ngrok

repo = Path('/content/ai-assisted-parallel-image-processing')
core_cli = repo / 'build-colab' / 'image_pipeline_cli'
if not core_cli.is_file():
    raise FileNotFoundError('Chưa có image_pipeline_cli. Hãy chạy cell setup/build trước.')

if '_pixel_lab_process' in globals() and _pixel_lab_process.poll() is None:
    _pixel_lab_process.terminate()
    try:
        _pixel_lab_process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        _pixel_lab_process.kill()
if '_pixel_lab_log' in globals() and not _pixel_lab_log.closed:
    _pixel_lab_log.close()

environment = os.environ.copy()
environment['PIP_CORE_CLI'] = str(core_cli)
log_path = Path('/tmp/pixel_lab_streamlit.log')
_pixel_lab_log = log_path.open('w', encoding='utf-8')
_pixel_lab_process = subprocess.Popen(
    [sys.executable, '-m', 'streamlit', 'run', 'app/app.py',
     '--server.address=0.0.0.0', '--server.port=8501',
     '--server.headless=true', '--server.enableCORS=false',
     '--server.enableXsrfProtection=false'],
    cwd=repo, env=environment, stdout=_pixel_lab_log,
    stderr=subprocess.STDOUT,
)

for _ in range(30):
    if _pixel_lab_process.poll() is not None:
        _pixel_lab_log.flush()
        raise RuntimeError(log_path.read_text(encoding='utf-8', errors='replace'))
    try:
        if requests.get('http://127.0.0.1:8501/_stcore/health', timeout=1).ok:
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    _pixel_lab_process.terminate()
    _pixel_lab_log.flush()
    raise TimeoutError(log_path.read_text(encoding='utf-8', errors='replace'))

try:
    from google.colab import userdata
    ngrok_token = userdata.get('NGROK_AUTHTOKEN')
except Exception:
    ngrok_token = getpass.getpass('Nhập ngrok authtoken (ký tự sẽ được ẩn): ')
if not ngrok_token:
    raise ValueError('Thiếu NGROK_AUTHTOKEN. Hãy thêm token trong Colab Secrets rồi chạy lại cell.')

ngrok.set_auth_token(ngrok_token)
del ngrok_token
ngrok.kill()
_pixel_lab_tunnel = ngrok.connect(8501, bind_tls=True)
ui_url = _pixel_lab_tunnel.public_url
display(HTML(f'<a href="{ui_url}" target="_blank" style="font-size:18px">Mở Pixel Lab Streamlit UI</a>'))
print('Server đang chạy. Log:', log_path)
print('Khi dùng xong, chạy ngrok.disconnect(ui_url) để đóng tunnel.')

## Lưu kết quả benchmark lên Drive (tùy chọn)

Chỉ mount Drive khi cần lưu kết quả. Nên benchmark trên ổ đĩa cục bộ `/content` rồi mới chép CSV/biểu đồ sang Drive để I/O mạng không làm sai lệch thời gian.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import shutil

source = Path('/content/ai-assisted-parallel-image-processing/benchmarks/results')
destination = Path('/content/drive/MyDrive/parallel-image-processing/results')
if source.exists():
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source, destination, dirs_exist_ok=True)
    print(f'Copied results to {destination}')
else:
    print('No benchmark results yet.')